# STEP 1 — INSTALL DEPENDENCIES

In [ ]:
!pip install -q     pymupdf     sentence-transformers     faiss-cpu     openai

print("✅ Dependencies installed")

# STEP 2 — IMPORT LIBRARIES

In [ ]:
import fitz
import re
import faiss
import numpy as np

from google.colab import files
from sentence_transformers import SentenceTransformer
from openai import OpenAI

print("✅ Libraries imported")

# STEP 3 — UPLOAD PDF FILES

In [ ]:
uploaded = files.upload()

pdf_files = list(uploaded.keys())

print(f"✅ Uploaded {len(pdf_files)} PDF(s)")

# STEP 4 — EXTRACT TEXT FROM PDFs

In [ ]:
documents = []

for pdf in pdf_files:

    doc = fitz.open(pdf)

    text = ""

    for page in doc:
        text += page.get_text()

    documents.append({
        "source": pdf,
        "text": text
    })

print(f"✅ Extracted text from {len(documents)} PDF(s)")

# STEP 5 — CHUNK DOCUMENTS

In [ ]:
def chunk_text(text, chunk_size=1000, overlap=200):

    text = re.sub(r'\s+', ' ', text)

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks


all_chunks = []
all_metadata = []

for doc in documents:

    chunks = chunk_text(doc["text"])

    for chunk in chunks:

        all_chunks.append(chunk)

        all_metadata.append({
            "source": doc["source"]
        })

print(f"✅ Created {len(all_chunks)} chunks")

# STEP 6 — CREATE EMBEDDINGS + FAISS INDEX

In [ ]:
print("⏳ Loading embedding model...")

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("⏳ Creating embeddings...")

embeddings = embedder.encode(
    all_chunks,
    show_progress_bar=True
)

embeddings = np.array(embeddings).astype("float32")

faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("✅ FAISS index ready")
print(f"📚 Indexed chunks: {len(all_chunks)}")

# STEP 7 — OPENROUTER SETUP

In [ ]:
OPENROUTER_API_KEY = input("Paste OpenRouter API Key: ")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

print("✅ OpenRouter initialized")

# STEP 8 — RAG QUERY FUNCTION

In [ ]:
def query_rag(question, n_results=5, verbose=True):

    query_embedding = np.array(
        embedder.encode([question])
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        n_results
    )

    context_parts = []

    for idx in indices[0]:

        if idx != -1:

            source = all_metadata[idx]["source"]

            chunk = all_chunks[idx]

            context_parts.append(
                f"[SOURCE: {source}]\n{chunk}"
            )

    context = "\n\n---\n\n".join(context_parts)

    if verbose:

        retrieved_sources = list(set([
            all_metadata[idx]["source"]
            for idx in indices[0]
            if idx != -1
        ]))

        print("\n📄 Retrieved from:")

        for src in retrieved_sources:
            print(f"  - {src}")

    prompt = f"""
You are a scientific cfRNA research assistant.

Use ONLY the provided context to answer.

If the answer is uncertain, say so.

CONTEXT:
{context}

QUESTION:
{question}

Provide:
- direct answer
- biological interpretation
- concise scientific explanation
"""

    response = client.chat.completions.create(

        model="meta-llama/llama-3.3-70b-instruct:free",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0.2,
        max_tokens=1000
    )

    answer = response.choices[0].message.content

    return answer

# STEP 9 — TEST QUESTION

In [ ]:
question = "What biomarkers are associated with lung cancer in cfRNA studies?"

answer = query_rag(question)

print("\n🧠 ANSWER:\n")
print(answer)

# STEP 10 — INTERACTIVE CHAT

In [ ]:
while True:

    question = input("\nAsk a question (or type exit): ")

    if question.lower() == "exit":

        print("👋 Exiting")
        break

    try:

        answer = query_rag(question)

        print("\n🧠 ANSWER:\n")
        print(answer)

    except Exception as e:

        print(f"\n❌ Error: {e}")